# Project
---
AI Resume & Cover Letter Generator


Project Description

---
Title: AI-Powered Resume and CV Generator with Google Drive Integration (Google Colab)

Objective:
To create an interactive Google Colab-based resume/CV generator that collects user input via prompts, dynamically creates a professional resume in both PDF and image (JPG) formats, and optionally uploads the result to the user's Google Drive. This project also supports image embedding (like a profile photo) and a choice between simple and "fancy" template designs.

Key Features:

Collects comprehensive resume data: personal info, education, work experience, skills, and additional sections (projects, awards, etc.).

Offers option to upload and embed a profile photo.

Supports both simple and styled (fancy) PDF templates.

Converts the final PDF to an image (JPG) using ImageMagick.

Allows download of generated files in PDF or JPG format.

Optionally uploads generated resume/image to Google Drive under a dedicated folder.

User-friendly prompts for all interactions — no manual coding needed by end-users.

Use Cases:

Students or job-seekers who need a quick, customizable resume builder.

People without access to professional resume tools like Canva, Overleaf, or MS Word.

Individuals interested in embedding profile photos and downloading resumes in image format for platforms like LinkedIn or job portals.

Implementation Details

---
| Tool/Library                                      | Purpose                                             |
| ---------------------------------    | --------------------------------------------------- |
| `google-auth` & `googleapiclient` | For authenticating and uploading to Google Drive    |
| `reportlab`                                         |  To create and format PDF files                      |
| `PIL` (Pillow)                                     | Image handling (used for profile picture inclusion) |
| `ImageMagick` (`convert`)             | Convert PDF to JPG image (via command-line)         |
| `Google Colab files`                        | To upload files and allow user downloads            |



Code


In [ ]:
fdff
fffffffffffffsssv


# Install necessary libraries
!pip install reportlab pillow

# Import libraries
import io
import os
import json
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from PIL import Image
import pathlib
import shutil

# --- User Input ---

print("--- Resume Information ---")

name = input("Full Name: ")
contact_info = input("Contact Information (Phone, Email, LinkedIn URL): ")
summary = input("Professional Summary: ")

print("\n--- Education ---")
education_entries = []
while True:
    degree = input("Degree (or 'done' to finish education): ")
    if degree.lower() == 'done':
        break
    institution = input("Institution: ")
    years = input("Years Attended: ")
    education_entries.append({'degree': degree, 'institution': institution, 'years': years})

print("\n--- Work Experience ---")
work_entries = []
while True:
    job_title = input("Job Title (or 'done' to finish work experience): ")
    if job_title.lower() == 'done':
        break
    company = input("Company: ")
    dates = input("Dates of Employment: ")
    responsibilities = []
    print("Enter key responsibilities (one per line, type 'done' when finished):")
    while True:
        resp = input()
        if resp.lower() == 'done':
            break
        responsibilities.append(resp)
    work_entries.append({'job_title': job_title, 'company': company, 'dates': dates, 'responsibilities': responsibilities})

print("\n--- Skills ---")
skills = input("Skills (comma-separated): ").split(',')

print("\n--- Additional Information (Optional) ---")
additional_sections = {}
while True:
    section_title = input("Additional Section Title (e.g., 'Projects', 'Awards', or 'done' to finish): ")
    if section_title.lower() == 'done':
        break
    content = []
    print(f"Enter content for '{section_title}' (one per line, type 'done' when finished):")
    while True:
        item = input()
        if item.lower() == 'done':
            break
        content.append(item)
    additional_sections[section_title] = content

print("\n--- Preferences ---")
upload_to_drive = False  # Replaced by GROQ local store option below
include_photo = input("Include a profile photo? (yes/no): ").lower() == 'yes'
use_fancy_template = input("Use a fancy template layout? (yes/no): ").lower() == 'yes'

# For local/GROQ usage: ask user for a local path to the profile photo (leave blank to skip)
profile_photo_path = None
if include_photo:
    profile_photo_path = input("Profile photo local path (leave blank to skip): ") or None
    if profile_photo_path and not os.path.exists(profile_photo_path):
        print("Provided photo path does not exist. Skipping photo.")
        profile_photo_path = None

# --- Data Structure for Resume ---
resume_data = {
    'name': name,
    'contact_info': contact_info,
    'summary': summary,
    'education': education_entries,
    'experience': work_entries,
    'skills': skills,
    'additional_sections': additional_sections
}

# --- Resume Generation (Simple Text-Based for now, will integrate templating later) ---

def generate_simple_resume_text(data):
    resume_text = f"{data['name']}\n{data['contact_info']}\n\n"
    resume_text += f"Summary:\n{data['summary']}\n\n"

    resume_text += "Education:\n"
    for entry in data['education']:
        resume_text += f"- {entry['degree']}, {entry['institution']}, {entry['years']}\n"
    resume_text += "\n"

    resume_text += "Work Experience:\n"
    for entry in data['experience']:
        resume_text += f"- {entry['job_title']} at {entry['company']} ({entry['dates']})\n"
        for resp in entry['responsibilities']:
            resume_text += f"  - {resp}\n"
        resume_text += "\n"

    resume_text += f"Skills:\n{', '.join(data['skills'])}\n\n"

    for title, content in data['additional_sections'].items():
        resume_text += f"{title}:\n"
        for item in content:
            resume_text += f"- {item}\n"
        resume_text += "\n"

    return resume_text.strip()

# --- PDF Generation ---

def generate_pdf(data, profile_photo_path=None, use_fancy_template=False):
    pdf_buffer = io.BytesIO()
    c = canvas.Canvas(pdf_buffer, pagesize=letter)

    if use_fancy_template:
        c.setFont("Helvetica-Bold", 16)
        c.drawString(100, 750, data['name'])
        c.setFont("Helvetica", 10)
        c.drawString(100, 735, data['contact_info'])

        y_position = 700

        if profile_photo_path and os.path.exists(profile_photo_path):
             try:
                 img = ImageReader(profile_photo_path)
                 img_width, img_height = img.getSize()
                 aspect_ratio = img_height / img_width
                 new_width = 80
                 new_height = new_width * aspect_ratio
                 c.drawImage(img, 480, y_position - new_height, width=new_width, height=new_height, mask='auto')
                 y_position -= new_height + 20
             except Exception as e:
                 print(f"Could not draw image: {e}")

        c.setFont("Helvetica-Bold", 12)
        c.drawString(100, y_position, "Summary")
        c.setFont("Helvetica", 10)
        y_position -= 15
        for line in data['summary'].split('\n'):
            c.drawString(120, y_position, line)
            y_position -= 12
        y_position -= 10

        c.setFont("Helvetica-Bold", 12)
        c.drawString(100, y_position, "Education")
        c.setFont("Helvetica", 10)
        y_position -= 15
        for entry in data['education']:
            c.drawString(120, y_position, f"{entry['degree']}, {entry['institution']}, {entry['years']}")
            y_position -= 12
        y_position -= 10

        c.setFont("Helvetica-Bold", 12)
        c.drawString(100, y_position, "Work Experience")
        c.setFont("Helvetica", 10)
        y_position -= 15
        for entry in data['experience']:
            c.drawString(120, y_position, f"{entry['job_title']} at {entry['company']} ({entry['dates']})")
            y_position -= 12
            for resp in entry['responsibilities']:
                c.drawString(140, y_position, f"- {resp}")
                y_position -= 12
            y_position -= 5
        y_position -= 10

        c.setFont("Helvetica-Bold", 12)
        c.drawString(100, y_position, "Skills")
        c.setFont("Helvetica", 10)
        y_position -= 15
        c.drawString(120, y_position, ', '.join(data['skills']))
        y_position -= 12
        y_position -= 10

        for title, content in data['additional_sections'].items():
            c.setFont("Helvetica-Bold", 12)
            c.drawString(100, y_position, title)
            c.setFont("Helvetica", 10)
            y_position -= 15
            for item in content:
                 c.drawString(120, y_position, f"- {item}")
                 y_position -= 12
            y_position -= 10


    else: # Simple template
        c.setFont("Helvetica", 12)
        text = c.beginText(50, 750)
        text.textLines(generate_simple_resume_text(data))
        c.drawText(text)

        if profile_photo_path and os.path.exists(profile_photo_path):
             try:
                 img = ImageReader(profile_photo_path)
                 img_width, img_height = img.getSize()
                 aspect_ratio = img_height / img_width
                 new_width = 80
                 new_height = new_width * aspect_ratio
                 c.drawImage(img, 480, 750 - new_height, width=new_width, height=new_height, mask='auto')
             except Exception as e:
                 print(f"Could not draw image: {e}")

    c.save()
    pdf_buffer.seek(0)
    return pdf_buffer

# --- Image Generation (from PDF) ---

def pdf_to_image(pdf_buffer):
    try:
        with open("temp_resume.pdf", "wb") as f:
            f.write(pdf_buffer.getvalue())
        # Attempt ImageMagick convert if available on system; this may fail on Windows without ImageMagick installed
        convert_cmd = 'convert -density 150 temp_resume.pdf -quality 90 resume.jpg'
        try:
            os.system(convert_cmd)
            if os.path.exists('resume.jpg'):
                os.remove('temp_resume.pdf')
                return 'resume.jpg'
        except Exception:
            pass
        # Fallback: leave PDF only
        if os.path.exists('temp_resume.pdf'):
            os.remove('temp_resume.pdf')
        return None

    except Exception as e:
        print(f"Could not convert PDF to Image. Error: {e}")
        return None

# --- GROQ (local) Store Integration ---

def upload_to_groq_store(file_path, namespace='MyResumeCVs'):
    """Simulate storing the file in a local GROQ-backed store by copying
    the file to a dedicated folder (`groq_store/namespace`)."""
    try:
        dest_dir = os.path.join('groq_store', namespace)
        pathlib.Path(dest_dir).mkdir(parents=True, exist_ok=True)
        dest_path = os.path.join(dest_dir, os.path.basename(file_path))
        shutil.copy2(file_path, dest_path)
        print(f"Stored '{file_path}' to GROQ store at: {dest_path}")
        return dest_path
    except Exception as e:
        print(f"Failed to store file to GROQ store: {e}")
        return None

# --- Main Execution ---

# Generate PDF
pdf_buffer = generate_pdf(resume_data, profile_photo_path, use_fancy_template)
pdf_filename = "resume.pdf"
with open(pdf_filename, "wb") as f:
    f.write(pdf_buffer.getvalue())

print(f"\nGenerated {pdf_filename}")

# Generate Image (from PDF)
image_filename = pdf_to_image(pdf_buffer)
if image_filename:
    print(f"Generated {image_filename}")

# --- Download / Store Options ---
print("\n--- Download / Store Options ---")
print(f"PDF saved locally at: {os.path.abspath(pdf_filename)}")
if image_filename:
    print(f"Image saved locally at: {os.path.abspath(image_filename)}")

# Local store prompts
save_pdf = input("Save PDF to GROQ store? (yes/no): ").lower() == 'yes'
if save_pdf:
    upload_to_groq_store(pdf_filename)

if image_filename:
    save_img = input("Save JPG to GROQ store? (yes/no): ").lower() == 'yes'
    if save_img:
        upload_to_groq_store(image_filename)

print("\nResume/CV generation complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.6 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,106 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,757 kB]
Get:1

Challenges and Solution

---

✅ Challenge 1: Collecting Structured User Input in a CLI-based Environment
Problem: Google Colab doesn’t offer rich UI components (like dropdowns or forms), so gathering user data becomes tedious with many input() prompts.

Solution:

Used loop-based prompts for flexible sections like education and experience.

Allowed users to type 'done' when finished to end input gracefully.

Ensured all fields are grouped logically (e.g., summary, then education, then work experience).

✅ Challenge 2: Formatting PDFs with a Professional Look
Problem: Using reportlab for layout can be tedious, especially when content overflows or has to be aligned with images (like profile photo).

Solution:

Created two modes: Simple template and Fancy layout.

In fancy mode:

Used drawString() and drawImage() to position text and photo manually.

Used vertical positioning with a y_position tracker to avoid text overlap.

Modularized layout blocks (summary, education, etc.) to handle dynamic content.

✅ Challenge 3: Embedding a Profile Photo Dynamically
Problem: Users might upload different image types/sizes, and positioning them properly in the PDF layout was difficult.

Solution:

Used ImageReader from reportlab.lib.utils to read the uploaded image.

Calculated aspect ratio to resize the photo proportionally.

Placed the image in a dedicated corner with fallback error handling (try-except).

✅ Challenge 4: Converting PDF to JPG for Platforms that Don't Accept PDFs
Problem: Google Colab doesn’t natively support PDF-to-JPG conversion, and libraries like wand need external dependencies.

Solution:

Installed ImageMagick (convert) using !apt-get install.

Saved the generated PDF temporarily and called:

bash
Copy
Edit
convert -density 150 temp_resume.pdf -quality 90 resume.jpg
Cleaned up temporary files after conversion.

✅ Challenge 5: Google Drive API Authentication and Upload
Problem: Uploading to a specific folder on Google Drive required proper authentication and folder-check logic.

Solution:

Used google.colab.auth for seamless OAuth2 authentication.

Checked for an existing folder named 'MyResumeCVs', or created one using:

python
Copy
Edit
service.files().create(body=file_metadata, fields='id')
Uploaded the final .pdf and .jpg with proper MIME type settings.

✅ Challenge 6: Making Output Downloadable from Google Colab
Problem: Files generated inside Colab can’t be directly accessed unless provided via the UI or download link.

Solution:

Used files.download() to trigger file downloads for both formats.

Displayed messages to guide users on when and what files were being downloaded.

✅ Challenge 7: Handling User Errors & Fallbacks
Problem: Users might forget to upload a photo, leave fields empty, or input invalid data.

Solution:

Handled optional photo uploads with a conditional if uploaded check.

Displayed fallback messages and continued without crashing.

Wrapped key image operations and Drive uploads in try-except blocks to catch errors.